In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go
from scripts import FDD as fdd

In [2]:
nodes = []
nodal_values = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')
for i in range(nodal_values.shape[0]):
  nodes.append(n.nodes(nodal_values[i, 0], nodal_values[i, 1], nodal_values[i, 2]))

In [3]:
E = 210e9  # Young's modulus in Pascals
nu = 0.3   # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus in Pascals
m = 6.7504 * 10 ** 6 #kg
m = 5596024
d1 = 0.8 #m
d2 = 1.8 #m
t1 = 0.03 #m
t2 = 0.08 #m
L_truss_element_y = 15 #m
h_truss = 18 #m
Aeq, Ieqy, Ieqz, b_eq, h_eq = mat.effective_truss_stiffness(d1, d2, t1, t2, h_truss, L_truss_element_y) 
L = 237.5
rho_truss = m / (L * Aeq)
It = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
k = 0.08
Ip  = Ieqy + Ieqz
ep_K = [E, G, Aeq, Ieqy, Ieqz, It, k]
ep_m = [rho_truss, Aeq, Ieqy, Ieqz, Ip]

In [4]:
k = 5/6
rho = 7850 #kg/m^3
E=210e9 #Pa
G = E/(2*(1+nu)) #Pa
nu = 0.3 #Poisson's ratio
k_fender =  mat.stiffness_fenders()
Iy_connect, Iz_connect, Ip_connect, It_connect, A_connect = mat.stiffness_connecting_beams()
ep_K_connect = [E, G, A_connect, Iy_connect, Iz_connect, It_connect, k]
ep_m_connect = [rho, A_connect, Iy_connect, Iz_connect, Ip_connect]

h_eq = 22
Iy_wall = 1250
Iz_wall = 0.25 * Iy_wall
Ip_wall = Iy_wall + Iz_wall
It_wall = 0.05 * Iy_wall
A_wall = Iy_wall * 12 / h_eq**2
m_wall = 6455148
L_wall = 237.5
rho_wall = m_wall / (L_wall * A_wall)
ep_K_wall = [E, G, A_wall, Iy_wall, Iz_wall, It_wall, k]
ep_m_wall = [rho_wall, A_wall, Iy_wall, Iz_wall, Ip_wall]


In [5]:

A_eq, Iy_eq, Iz_eq, b_eq, h_eq = mat.stiffness_connecting_truss(d1, d2, t1, t2, h_truss, L_truss_element_y)
Ip_connecting_truss = Iy_eq + Iz_eq 
It_eq = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
ep_K_connecting_truss = [E, G, A_eq, Iy_eq, Iz_eq, It_eq, k]
ep_m_connecting_truss = [rho_truss, A_eq, Iy_eq, Iz_eq, Ip_connect]

In [6]:
k = 5/6
Iy, Iz, Ip, It, A = mat.stiffness_braces()
ep_K_braces = [E, G, A, Iy, Iz, It, k]
ep_m_braces = [rho, A, Iy, Iz, Ip]

In [7]:
elements = []
element_nodes = np.loadtxt('../text_files/element_nodes.txt', dtype=int)
for i in range(element_nodes.shape[0]):
    if element_nodes[i, 2] == 0:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K, ep_m))
    elif element_nodes[i, 2] == 1:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_wall, ep_m_wall))
    elif element_nodes[i, 2] == 2:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_connect, ep_m_connect))
    elif element_nodes[i, 2] == 3:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_connecting_truss, ep_m_connecting_truss))   
    elif element_nodes[i, 2] == 4:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_braces, ep_m_braces))
element_nodes = element_nodes[:, :2] # Remove the column with element type information


dofs = n.degrees_of_freedom(nodes)

element_locs = []

for (nA, nB) in element_nodes:
    dofs_A = dofs[f'dof_{nA}']
    dofs_B = dofs[f'dof_{nB}']
    element_locs.append(np.hstack((dofs_A, dofs_B)))



In [11]:
eigvecs_experiment = np.load('experimental_eigvectors/phi_experiment_22_sensors.npy')
eigvecs_experiment_12_sensors = np.load('experimental_eigvectors/phi_experiment_12_sensors.npy')
eigvecs_experiment_2_sensors = np.load('experimental_eigvectors/phi_experiment_2_sensors.npy')  
node_numbers = [1, 2, 3, 4, 8, 12, 16, 19, 21, 27, 31, 35 , 
                38, 40, 46, 49, 55, 58, 70, 75, 94, 113]
node_numbers_retainingwall = [3, 4, 8, 12, 16, 19, 21, 27, 31, 35, 38, 40, 46, 49, 55, 58]
node_numbers_primarytruss = [1, 2, 70, 75, 94, 113]

## **Mode shapes using 22 sensors**

In [ ]:
nodes_all = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')[:, :3]
eigvecs_experiment = np.asarray(eigvecs_experiment, dtype=np.complex128).real

sensor_coords = nodes_all[node_numbers]
num_sensors = len(node_numbers)

node_to_index = {node_id: i for i, node_id in enumerate(node_numbers)}


def get_mode(mode_idx, scale):
    mode_vector = eigvecs_experiment[:, mode_idx].real
    mode_xyz = mode_vector.reshape(num_sensors, 3)

    U0 = sensor_coords
    U = U0 + scale * mode_xyz
    return U0, U



modes_to_plot = 4

default_scale = 80
default_zoom_factor = 0.9

scale_list = [default_scale, default_scale, default_scale, 70] 
zoom_factor_list = [default_zoom_factor, default_zoom_factor, default_zoom_factor, 2.5]  


node_numbers_primarytruss = np.array([
    3, 4, 8, 12, 16, 19, 21, 27, 31, 35,
    38, 40, 46, 49, 55, 58
])

node_numbers_retainingwall = np.array([
    1, 2, 70, 75, 94, 113
])

retaining_mask = np.isin(node_numbers, node_numbers_retainingwall)
truss_mask = np.isin(node_numbers, node_numbers_primarytruss)


chain_green_1 = np.array([8, 12, 16, 19, 21, 4, 46, 49, 1])
chain_green_2 = np.array([27, 31, 35, 38, 40, 3, 55, 58, 2])
chain_red_1   = np.array([70, 75, 1, 94, 2, 113])

def to_index_chain(chain):
    return np.array([node_to_index[n] for n in chain if n in node_to_index])


min_x, max_x = sensor_coords[:, 0].min(), sensor_coords[:, 0].max()
min_y, max_y = sensor_coords[:, 1].min(), sensor_coords[:, 1].max()
min_z, max_z = sensor_coords[:, 2].min(), sensor_coords[:, 2].max()

Lmax = max(max_x - min_x, max_y - min_y, max_z - min_z)

mid_x = (min_x + max_x) / 2
mid_y = (min_y + max_y) / 2
mid_z = (min_z + max_z) / 2

plane_margin = 0.6 * Lmax

x_grid = np.linspace(mid_x - plane_margin, mid_x + plane_margin, 15)
y_grid = np.linspace(mid_y - plane_margin, mid_y + plane_margin, 15)

Xg, Yg = np.meshgrid(x_grid, y_grid)
Zg = np.zeros_like(Xg)

plane = go.Surface(
    x=Xg,
    y=Yg,
    z=Zg,
    showscale=False,
    opacity=0.2,
    colorscale=[[0, "rgb(200,200,200)"], [1, "rgb(200,200,200)"]],
    hoverinfo="skip"
)

def add_chain(traces, chain_global, color, coords):
    chain_idx = to_index_chain(chain_global)

    for k in range(len(chain_idx) - 1):
        a = chain_idx[k]
        b = chain_idx[k + 1]

        traces.append(go.Scatter3d(
            x=[coords[a, 0], coords[b, 0]],
            y=[coords[a, 1], coords[b, 1]],
            z=[coords[a, 2], coords[b, 2]],
            mode='lines',
            line=dict(color=color, width=5),
            showlegend=False
        ))



for i in range(modes_to_plot):

    scale = scale_list[i]
    zoom_factor = zoom_factor_list[i]

    U0, U = get_mode(i, scale)

    traces = []

    min_x, max_x = sensor_coords[:, 0].min(), sensor_coords[:, 0].max()
    min_y, max_y = sensor_coords[:, 1].min(), sensor_coords[:, 1].max()
    min_z, max_z = sensor_coords[:, 2].min(), sensor_coords[:, 2].max()

    Lmax = max(max_x - min_x, max_y - min_y, max_z - min_z)

    mid_x = (min_x + max_x) / 2
    mid_y = (min_y + max_y) / 2
    mid_z = (min_z + max_z) / 2

    zoom = zoom_factor * Lmax

    xr = [mid_x - zoom, mid_x + zoom]
    yr = [mid_y - zoom, mid_y + zoom]
    zr = [mid_z - zoom, mid_z + zoom]

    traces.append(plane)

    add_chain(traces, chain_green_1, "lightgrey", U0)
    add_chain(traces, chain_green_2, "lightgrey", U0)
    add_chain(traces, chain_red_1,   "lightgrey", U0)

    traces.append(go.Scatter3d(
        x=U[retaining_mask, 0],
        y=U[retaining_mask, 1],
        z=U[retaining_mask, 2],
        mode='markers',
        marker=dict(size=4, color='green'),
        name='Retaining wall'
    ))

    traces.append(go.Scatter3d(
        x=U[truss_mask, 0],
        y=U[truss_mask, 1],
        z=U[truss_mask, 2],
        mode='markers',
        marker=dict(size=4, color='red'),
        name='Primary truss'
    ))

    add_chain(traces, chain_green_1, "red", U)
    add_chain(traces, chain_green_2, "red", U)
    add_chain(traces, chain_red_1,   "green", U)

    fig = go.Figure(data=traces)

    fig.update_layout(
        title=f"Mode Shape {i+1}",
        width=950,
        height=750,

        scene=dict(
            xaxis=dict(range=xr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            yaxis=dict(range=yr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            zaxis=dict(range=zr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),

            aspectmode='data',
            camera=dict(eye=dict(x=1.2, y=1.2, z=0.8))
        ),

        margin=dict(l=10, r=10, b=10, t=50)
    )

    fig.show()

## **Mode shapes using 12 sensors**

In [13]:
nodes_all = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')[:, :3]
eigvecs_experiment = np.asarray(eigvecs_experiment_12_sensors, dtype=np.complex128).real

node_numbers = [1, 2, 3, 4, 12, 19, 31, 38, 70, 75, 94, 113]
sensor_coords = nodes_all[node_numbers]
num_sensors = len(node_numbers)

node_to_index = {node_id: i for i, node_id in enumerate(node_numbers)}


def get_mode(mode_idx, scale):
    mode_vector = eigvecs_experiment[:, mode_idx].real
    mode_xyz = mode_vector.reshape(num_sensors, 3)

    U0 = sensor_coords
    U = U0 + scale * mode_xyz
    return U0, U


modes_to_plot = 4


default_scale = 80
default_zoom_factor = 0.9


scale_list = [default_scale, default_scale, default_scale, 70]   
zoom_factor_list = [default_zoom_factor, default_zoom_factor, default_zoom_factor, 2.5] 


node_numbers_primarytruss = np.array([
    3, 4, 12, 19, 31, 38
])

node_numbers_retainingwall = np.array([
    1, 2, 70, 75, 94, 113
])

retaining_mask = np.isin(node_numbers, node_numbers_retainingwall)
truss_mask = np.isin(node_numbers, node_numbers_primarytruss)


chain_green_1 = np.array([12, 19, 4, 1])
chain_green_2 = np.array([31, 38, 3, 2])
chain_red_1   = np.array([70, 75, 1, 94, 2, 113])

def to_index_chain(chain):
    return np.array([node_to_index[n] for n in chain if n in node_to_index])

for i in range(modes_to_plot):

    scale = scale_list[i]
    zoom_factor = zoom_factor_list[i]

    U0, U = get_mode(i, scale)

    traces = []

    min_x, max_x = sensor_coords[:, 0].min(), sensor_coords[:, 0].max()
    min_y, max_y = sensor_coords[:, 1].min(), sensor_coords[:, 1].max()
    min_z, max_z = sensor_coords[:, 2].min(), sensor_coords[:, 2].max()

    Lmax = max(max_x - min_x, max_y - min_y, max_z - min_z)

    mid_x = (min_x + max_x) / 2
    mid_y = (min_y + max_y) / 2
    mid_z = (min_z + max_z) / 2

    zoom = zoom_factor * Lmax

    xr = [mid_x - zoom, mid_x + zoom]
    yr = [mid_y - zoom, mid_y + zoom]
    zr = [mid_z - zoom, mid_z + zoom]

    plane_margin = 0.6 * Lmax
    x_grid = np.linspace(mid_x - plane_margin, mid_x + plane_margin, 15)
    y_grid = np.linspace(mid_y - plane_margin, mid_y + plane_margin, 15)
    Xg, Yg = np.meshgrid(x_grid, y_grid)
    Zg = np.zeros_like(Xg)

    plane = go.Surface(
        x=Xg,
        y=Yg,
        z=Zg,
        showscale=False,
        opacity=0.2,
        colorscale=[[0, "rgb(200,200,200)"], [1, "rgb(200,200,200)"]],
        hoverinfo="skip"
    )

    traces.append(plane)

    def add_chain(traces, chain_global, color, coords):
        chain_idx = to_index_chain(chain_global)

        for k in range(len(chain_idx) - 1):
            a = chain_idx[k]
            b = chain_idx[k + 1]

            traces.append(go.Scatter3d(
                x=[coords[a, 0], coords[b, 0]],
                y=[coords[a, 1], coords[b, 1]],
                z=[coords[a, 2], coords[b, 2]],
                mode='lines',
                line=dict(color=color, width=5),
                showlegend=False
            ))

    add_chain(traces, chain_green_1, "lightgrey", U0)
    add_chain(traces, chain_green_2, "lightgrey", U0)
    add_chain(traces, chain_red_1,   "lightgrey", U0)


    traces.append(go.Scatter3d(
        x=U[retaining_mask, 0],
        y=U[retaining_mask, 1],
        z=U[retaining_mask, 2],
        mode='markers',
        marker=dict(size=4, color='green'),
        name='Retaining wall'
    ))

    traces.append(go.Scatter3d(
        x=U[truss_mask, 0],
        y=U[truss_mask, 1],
        z=U[truss_mask, 2],
        mode='markers',
        marker=dict(size=4, color='red'),
        name='Primary truss'
    ))


    add_chain(traces, chain_green_1, "red", U)
    add_chain(traces, chain_green_2, "red", U)
    add_chain(traces, chain_red_1,   "green", U)


    fig = go.Figure(data=traces)

    fig.update_layout(
        title=f"Mode Shape {i+1}",
        width=950,
        height=750,

        scene=dict(
            xaxis=dict(range=xr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            yaxis=dict(range=yr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            zaxis=dict(range=zr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),

            aspectmode='data',
            camera=dict(eye=dict(x=1.2, y=1.2, z=0.8))
        ),

        margin=dict(l=10, r=10, b=10, t=50)
    )

    fig.show()

In [14]:
nodes_all = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')[:, :3]
eigvecs_experiment = np.asarray(eigvecs_experiment_2_sensors, dtype=np.complex128).real
node_numbers = [1, 38]
sensor_coords = nodes_all[node_numbers]
num_sensors = len(node_numbers)

node_to_index = {node_id: i for i, node_id in enumerate(node_numbers)}

def get_mode(mode_idx, scale=1.0):
    mode_vector = eigvecs_experiment[:, mode_idx].real
    mode_xyz = mode_vector.reshape(num_sensors, 3)

    U0 = sensor_coords
    U = U0 + scale * mode_xyz
    return U0, U

modes_to_plot = 4
scale = 80

node_numbers_primarytruss = np.array([
    38
])

node_numbers_retainingwall = np.array([
    1
])

retaining_mask = np.isin(node_numbers, node_numbers_retainingwall)
truss_mask = np.isin(node_numbers, node_numbers_primarytruss)

chain_green_1 = np.array([1])
chain_green_2 = np.array([38])
chain_red_1   = np.array([])

def to_index_chain(chain):
    return np.array([node_to_index[n] for n in chain if n in node_to_index])

min_x, max_x = sensor_coords[:, 0].min(), sensor_coords[:, 0].max()
min_y, max_y = sensor_coords[:, 1].min(), sensor_coords[:, 1].max()
min_z, max_z = sensor_coords[:, 2].min(), sensor_coords[:, 2].max()

Lmax = max(max_x - min_x, max_y - min_y, max_z - min_z)

mid_x = (min_x + max_x) / 2
mid_y = (min_y + max_y) / 2
mid_z = (min_z + max_z) / 2

zoom = 0.9 * Lmax 

xr = [mid_x - zoom, mid_x + zoom]
yr = [mid_y - zoom, mid_y + zoom]
zr = [mid_z - zoom, mid_z + zoom]


plane_margin = 0.6 * Lmax

x_grid = np.linspace(mid_x - plane_margin, mid_x + plane_margin, 15)
y_grid = np.linspace(mid_y - plane_margin, mid_y + plane_margin, 15)

Xg, Yg = np.meshgrid(x_grid, y_grid)
Zg = np.zeros_like(Xg)

plane = go.Surface(
    x=Xg,
    y=Yg,
    z=Zg,
    showscale=False,
    opacity=0.2,
    colorscale=[[0, "rgb(200,200,200)"], [1, "rgb(200,200,200)"]],
    hoverinfo="skip"
)

def add_chain(traces, chain_global, color, coords):
    chain_idx = to_index_chain(chain_global)

    for k in range(len(chain_idx) - 1):
        a = chain_idx[k]
        b = chain_idx[k + 1]

        traces.append(go.Scatter3d(
            x=[coords[a, 0], coords[b, 0]],
            y=[coords[a, 1], coords[b, 1]],
            z=[coords[a, 2], coords[b, 2]],
            mode='lines',
            line=dict(color=color, width=5),
            showlegend=False
        ))

for i in range(modes_to_plot):

    U0, U = get_mode(i, scale)

    traces = []

    traces.append(plane)

    add_chain(traces, chain_green_1, "lightgrey", U0)
    add_chain(traces, chain_green_2, "lightgrey", U0)
    add_chain(traces, chain_red_1,   "lightgrey", U0)

    traces.append(go.Scatter3d(
        x=U[retaining_mask, 0],
        y=U[retaining_mask, 1],
        z=U[retaining_mask, 2],
        mode='markers',
        marker=dict(size=4, color='green'),
        name='Retaining wall'
    ))

    traces.append(go.Scatter3d(
        x=U[truss_mask, 0],
        y=U[truss_mask, 1],
        z=U[truss_mask, 2],
        mode='markers',
        marker=dict(size=4, color='red'),
        name='Primary truss'
    ))

    add_chain(traces, chain_green_1, "red", U)
    add_chain(traces, chain_green_2, "red", U)
    add_chain(traces, chain_red_1,   "green",   U)

    fig = go.Figure(data=traces)

    fig.update_layout(
        title=f"Mode Shape {i+1}",
        width=950,
        height=750,

        scene=dict(
            xaxis=dict(range=xr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            yaxis=dict(range=yr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),
            zaxis=dict(range=zr, showbackground=False, showgrid=False,
                       zeroline=False, showticklabels=False),

            aspectmode='data',
            camera=dict(eye=dict(x=1.2, y=1.2, z=0.8))
        ),

        margin=dict(l=10, r=10, b=10, t=50)
    )

    fig.show()